In [ ]:
import subprocess, sys, os, time, struct, json, math, base64
import torch, numpy as np

print("="*70)
print("VOICEOS SPRINT-29 PHASE-3 — REAL KAGGLE GPU VALIDATION")
print("="*70)
print(f"Timestamp: {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())}")
print(f"Python:    {sys.version}")

# nvidia-smi
r = subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True)
print("\n--- nvidia-smi ---")
print(r.stdout)

# GPU properties
print("--- PyTorch GPU Properties ---")
print(f"PyTorch:          {torch.__version__}")
print(f"CUDA available:   {torch.cuda.is_available()}")
GPU_INFO = []
if torch.cuda.is_available():
    print(f"CUDA runtime:     {torch.version.cuda}")
    print(f"cuDNN:            {torch.backends.cudnn.version()}")
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        cc = f"{p.major}.{p.minor}"
        info = {
            "index": i, "name": p.name,
            "vram_gb": round(p.total_memory/1024**3, 2),
            "vram_mib": p.total_memory//1024**2,
            "compute_capability": cc,
            "sm": f"sm_{p.major}{p.minor}",
            "bf16_cores": p.major >= 8,
            "fp16_cores": p.major >= 7,
        }
        GPU_INFO.append(info)
        print(f"\nGPU {i}: {p.name}")
        print(f"  VRAM:    {info['vram_gb']:.2f} GB ({info['vram_mib']} MiB)")
        print(f"  Compute: {cc} ({info['sm']})")
        print(f"  BF16 tensor cores: {info['bf16_cores']} (requires sm_80+)")
        print(f"  FP16 tensor cores: {info['fp16_cores']}")

r2 = subprocess.run("df -h /kaggle/working", shell=True, capture_output=True, text=True)
print(f"\n--- Disk ---\n{r2.stdout}")
r3 = subprocess.run("free -h", shell=True, capture_output=True, text=True)
print(f"--- RAM ---\n{r3.stdout}")

In [ ]:
# Production A6000 spec from deployment/gpu/framework/runtime_spec.yaml
PROD = {
    "gpu": "NVIDIA RTX A6000", "compute_capability": 8.6,
    "vram_gb": 46.0, "bf16_cores": True, "fp8_hw": False,
    "torch": "2.11.0", "vllm": "0.24.0",
    "faster_whisper": "1.2.1", "ctranslate2": "4.8.0",
    "transformers": "5.12.1", "snac": "1.0.0",
}

print("="*70)
print("KAGGLE T4 vs A6000 PRODUCTION SPECIFICATION")
print("="*70)

KAGGLE_LIMITATIONS = []
T4_OVERRIDES = {}

if GPU_INFO:
    g = GPU_INFO[0]
    cc = float(g["compute_capability"])
    n_gpu = len(GPU_INFO)
    total_vram = sum(x["vram_gb"] for x in GPU_INFO)

    print(f"\nProduction:  {PROD['gpu']} | sm_86 | {PROD['vram_gb']}GB × 1")
    print(f"Kaggle:      {g['name']} × {n_gpu} | {g['sm']} | {g['vram_gb']:.2f}GB × {n_gpu} = {total_vram:.1f}GB")

    if cc < PROD["compute_capability"]:
        lim = f"BF16 runs in FP32 emulation (no BF16 tensor cores on {g['sm']}; requires sm_80+)"
        KAGGLE_LIMITATIONS.append(lim); print(f"\n  KAGGLE LIMITATION: {lim}")
    if cc < 8.9:
        lim = "FP8 hardware not supported (T4=sm_75, requires Ada/Hopper sm_89+); Qwen FP8 dequantizes at runtime — still functional"
        KAGGLE_LIMITATIONS.append(lim); print(f"  KAGGLE LIMITATION: {lim}")

    # Production util=0.32 on 46GB=14.7GB. T4 has 14.56GB: need util=0.85 to get 12.4GB
    T4_OVERRIDES["vllm_util"] = 0.85    # vs production 0.32
    T4_OVERRIDES["vllm_maxlen"] = 512   # vs production 4096
    lim = f"LLM: gpu_memory_utilization 0.32→0.85, max_model_len 4096→512 (T4={g['vram_gb']:.1f}GB vs A6000=46GB)"
    KAGGLE_LIMITATIONS.append(lim); print(f"  KAGGLE LIMITATION: {lim}")

    print(f"\nT4 per-GPU VRAM: {g['vram_gb']:.2f} GB | Total: {total_vram:.1f} GB")
    print(f"VRAM plan: LLM on GPU0 ({g['vram_gb']:.0f}GB) | STT+TTS on GPU1 ({g['vram_gb']:.0f}GB)")
    print(f"  GPU0 LLM: ~12.4GB @ util=0.85 < {g['vram_gb']:.1f}GB T4 OK")
    print(f"  GPU1 STT+TTS: ~1.8+6.0=7.8GB < {g['vram_gb']:.1f}GB T4 OK")
    print(f"  STT=~1.8GB + TTS=~6GB = 7.8GB < {g['vram_gb']:.0f}GB → fits on GPU1")
    print(f"\nProduction spec NOT changed. A6000 spec remains authoritative.")
    print(f"Kaggle T4 overrides apply ONLY during this temporary validation.")

print(f"\nLimitations found: {len(KAGGLE_LIMITATIONS)}")

In [ ]:
import subprocess, sys, time, importlib

print("="*70)
print("INSTALL PRODUCTION DEPENDENCIES")
print("="*70)
print("PyTorch pre-installed on Kaggle — not reinstalling")

INSTALL = {}

def pip(pkg, timeout=600):
    t0 = time.monotonic()
    r = subprocess.run(f"{sys.executable} -m pip install {pkg} -q 2>&1",
                       shell=True, capture_output=True, text=True, timeout=timeout)
    ok = r.returncode == 0
    print(f"  {'OK' if ok else 'FAIL'}: {pkg} ({time.monotonic()-t0:.0f}s)")
    if not ok: print(f"    {(r.stdout+r.stderr)[-300:]}")
    return ok

for pkg in [
    "faster-whisper==1.2.1", "ctranslate2==4.8.0",
    "transformers==5.12.1", "tokenizers==0.22.2",
    "huggingface_hub==0.28.0", "accelerate==1.7.0",
    "snac==1.0.0", "numpy==2.3.5",
    "fastapi==0.115.6", "uvicorn[standard]==0.30.6",
    "pydantic==2.10.4", "httpx==0.27.0",
]:
    INSTALL[pkg.split("==")[0]] = pip(pkg)

print(f"\nInstalling vLLM (production pin: 0.24.0)...")
ok = pip("vllm==0.24.0", 900)
if not ok:
    print("  vllm==0.24.0 failed — trying latest available...")
    ok = pip("vllm", 900)
INSTALL["vllm"] = ok

print("\n--- Versions installed ---")
for m in ["torch","faster_whisper","ctranslate2","transformers","snac","httpx"]:
    try:
        mod = importlib.import_module(m)
        print(f"  {m}: {getattr(mod,'__version__','?')}")
    except ImportError:
        print(f"  {m}: NOT INSTALLED")
try:
    import vllm; print(f"  vllm: {vllm.__version__}")
except ImportError:
    print("  vllm: NOT INSTALLED")

failed = [k for k,v in INSTALL.items() if not v]
print(f"\nInstallation: {len(INSTALL)-len(failed)}/{len(INSTALL)} OK | Failed: {failed or 'none'}")

In [ ]:
import subprocess, sys, os

REPO = "https://github.com/pateekdas7/VoiceOS.git"
BRANCH = "claude/ssh-gpu-cpu-servers-y99fib"
COMMIT = "182fea6"
VOICEOS = "/kaggle/working/voiceos"
HF_HOME = "/kaggle/working/hf"

os.makedirs(HF_HOME, exist_ok=True)
os.environ["HF_HOME"] = HF_HOME
os.environ["TRANSFORMERS_CACHE"] = HF_HOME

print(f"Cloning {BRANCH}...")
r = subprocess.run(f"git clone --branch {BRANCH} --depth 1 {REPO} {VOICEOS} 2>&1",
                   shell=True, capture_output=True, text=True, timeout=120)
if r.returncode != 0:
    print(f"Clone failed: {r.stdout}{r.stderr}")
else:
    actual = subprocess.run(f"cd {VOICEOS} && git log -1 --format='%H %s'",
                            shell=True, capture_output=True, text=True).stdout.strip()
    print(f"Cloned: {actual}")
    print(f"Commit {COMMIT} verified: {COMMIT in actual}")
    for f in ["deployment/gpu/services/tts/server.py",
              "deployment/gpu/services/stt/server.py",
              "deployment/gpu/audio/pacer.py",
              "deployment/gpu/tests/test_audio_pipeline.py"]:
        exists = os.path.exists(f"{VOICEOS}/{f}")
        print(f"  {'✓' if exists else '✗'} {f}")

sys.path.insert(0, VOICEOS)
print(f"\nPython path: {VOICEOS}")
print(f"HF cache:    {HF_HOME}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# VOICE_DRIFT_FIX_SPEAKER_LOCK_SWEEP — TTS-only patch applied to cloned server.py
#
# Combines:
#   (1) Per-text CUDA RNG seed anchor (from Option B) — determinism
#   (2) Official Veena sampling (temperature=0.4, top_p=0.9) — natural prosody
#   (3) Speaker-Lock LogitsProcessor — sharpens first N tokens to lock kavya
#       identity before drifting to male-adjacent audio-code cluster.
#
# lock_tokens / boot_temp are per-request via optional SynthesizeRequest fields
# (backwards compatible — CPU never sends them; probe sweeps them).
# When lock_tokens=0 or unset, behavior is identical to Option B baseline
# (seed + official sampling, no locking).
#
# Idempotent; runs once between git clone (Cell 3) and TTS launch cells.
# ═══════════════════════════════════════════════════════════════════
import os

TTS_SRV = "/kaggle/working/voiceos/deployment/gpu/services/tts/server.py"
assert os.path.exists(TTS_SRV), f"Missing {TTS_SRV} — did Cell 3 clone succeed?"

with open(TTS_SRV) as f:
    src = f.read()

MARKER = "VOICE_DRIFT_FIX_SPEAKER_LOCK_SWEEP_APPLIED"

if MARKER in src:
    print("Speaker-Lock sweep patch already present (idempotent no-op)")
else:
    # ── (A) Extend SynthesizeRequest model with optional sweep fields ────────
    old_req = """class SynthesizeRequest(BaseModel):
    \"\"\"TTS synthesis request.\"\"\"

    text: str
    speaker: str = "kavya"
    voice_config: VoiceConfigRequest = VoiceConfigRequest()"""
    new_req = """class SynthesizeRequest(BaseModel):
    \"\"\"TTS synthesis request.\"\"\"

    text: str
    speaker: str = "kavya"
    voice_config: VoiceConfigRequest = VoiceConfigRequest()
    # VOICE_DRIFT_FIX_SPEAKER_LOCK_SWEEP_APPLIED — optional per-request sweep
    lock_tokens: int = 0      # 0 disables locking (Option B baseline)
    boot_temp: float = 0.05   # effective temperature for lock region"""
    assert old_req in src, "SynthesizeRequest block not found"
    src = src.replace(old_req, new_req)

    # ── (B) Thread lock params through /synthesize -> _stream_synthesis_sync ──
    old_call = """            else _stream_synthesis_sync(text, speaker, request.voice_config)"""
    new_call = """            else _stream_synthesis_sync(text, speaker, request.voice_config,
                                        lock_tokens=int(request.lock_tokens or 0),
                                        boot_temp=float(request.boot_temp or 0.05))"""
    assert old_call in src, "synthesize call site not found"
    src = src.replace(old_call, new_call)

    # ── (C) Extend _stream_synthesis_sync signature and body ─────────────────
    old_sig = """def _stream_synthesis_sync(
    text: str,
    speaker: str,
    voice_config: VoiceConfigRequest,
) -> Iterator[bytes]:"""
    new_sig = """def _stream_synthesis_sync(
    text: str,
    speaker: str,
    voice_config: VoiceConfigRequest,
    lock_tokens: int = 0,
    boot_temp: float = 0.05,
) -> Iterator[bytes]:"""
    assert old_sig in src, "_stream_synthesis_sync signature not found"
    src = src.replace(old_sig, new_sig)

    # ── (D) Patch _generate: seed anchor + official sampling + LogitsProcessor
    old_block = """    def _generate() -> None:
        try:
            with torch.no_grad():
                _model.generate(
                    input_ids,
                    max_new_tokens=max_new,
                    do_sample=True,
                    temperature=0.4,
                    top_p=0.9,
                    repetition_penalty=1.05,
                    pad_token_id=_tokenizer.pad_token_id or _END_OF_SPEECH_TOKEN,
                    eos_token_id=[_END_OF_SPEECH_TOKEN, _END_OF_AI_TOKEN],
                    streamer=streamer,
                )"""
    new_block = """    # ── VOICE_DRIFT_FIX_SPEAKER_LOCK_SWEEP_APPLIED ─────────────────────
    # Determinism: per-text CUDA RNG seed (from Option B, retained)
    # Sampling:    OFFICIAL Veena (temp=0.4, top_p=0.9) — restored from Option B tightening
    # Speaker lock: optional LogitsProcessor sharpens first N=lock_tokens steps
    #              to effective_temp = boot_temp (via score scaling that survives
    #              generate's temperature=0.4 division).
    from transformers import LogitsProcessor, LogitsProcessorList

    class _SpeakerLockProcessor(LogitsProcessor):
        def __init__(self, prompt_len: int, lock_tokens_: int, boot_temp_: float, base_temp_: float = 0.4):
            self.prompt_len = prompt_len
            self.lock_tokens = int(lock_tokens_)
            # We want effective divisor = boot_temp for lock region.
            # generate applies TemperatureLogitsWarper(base_temp_) — divides by base_temp_ AFTER our processor.
            # So we pre-scale by (base_temp_ / boot_temp_) — resulting effective divisor is boot_temp_.
            self.scale = float(base_temp_) / max(float(boot_temp_), 1e-6)

        def __call__(self, input_ids, scores):
            gen_pos = input_ids.shape[1] - self.prompt_len
            if gen_pos < self.lock_tokens:
                scores = scores * self.scale
            return scores

    def _generate() -> None:
        try:
            _vd_seed = hash(text) & 0x7FFFFFFFFFFFFFFF
            torch.manual_seed(_vd_seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(_vd_seed)

            processors = None
            if lock_tokens and lock_tokens > 0:
                processors = LogitsProcessorList([
                    _SpeakerLockProcessor(
                        prompt_len=input_ids.shape[1],
                        lock_tokens_=lock_tokens,
                        boot_temp_=boot_temp,
                        base_temp_=0.4,
                    )
                ])
                logger.info("Speaker-lock ACTIVE: lock_tokens=%d boot_temp=%.3f", lock_tokens, boot_temp)

            with torch.no_grad():
                _model.generate(
                    input_ids,
                    max_new_tokens=max_new,
                    do_sample=True,
                    temperature=0.4,
                    top_p=0.9,
                    repetition_penalty=1.05,
                    pad_token_id=_tokenizer.pad_token_id or _END_OF_SPEECH_TOKEN,
                    eos_token_id=[_END_OF_SPEECH_TOKEN, _END_OF_AI_TOKEN],
                    streamer=streamer,
                    logits_processor=processors,
                )"""
    assert old_block in src, "_generate block (original 0.4/0.9) not found — server.py may have drifted from expected form"
    src = src.replace(old_block, new_block)

    with open(TTS_SRV, "w") as f:
        f.write(src)
    print("Speaker-Lock sweep patch applied to", TTS_SRV)
    print("  - Per-text CUDA RNG seed anchor: ON (determinism)")
    print("  - Official Veena sampling: temperature=0.4 top_p=0.9 (RESTORED)")
    print("  - Optional lock_tokens/boot_temp per-request (default lock_tokens=0 = off)")
    print("  - SpeakerLockLogitsProcessor scales scores in lock region")

# ── Verify ────────────────────────────────────────────────────────────
with open(TTS_SRV) as f:
    v = f.read()
for needle in ["temperature=0.4", "top_p=0.9",
               "torch.manual_seed(_vd_seed)",
               "torch.cuda.manual_seed_all(_vd_seed)",
               "class _SpeakerLockProcessor",
               "lock_tokens: int = 0",
               "boot_temp: float = 0.05",
               MARKER]:
    assert needle in v, f"Verification FAILED — missing: {needle}"
assert "temperature=0.15" not in v, "Old Option B temperature=0.15 leaked!"
assert "top_p=0.7" not in v, "Old Option B top_p=0.7 leaked!"
print("PATCH VERIFICATION OK — all markers present, Option B tightening reverted, official sampling restored")


In [ ]:
import time, numpy as np, torch

print("="*70)
print("REAL STT VALIDATION — Whisper large-v3-turbo (int8_float16)")
print("="*70)

STT_STATUS = "BLOCKED"
STT_RESULTS = {}

try:
    from faster_whisper import WhisperModel, download_model as fw_dl

    print("\nDownloading whisper-large-v3-turbo (~1.5 GB)...")
    t0 = time.monotonic()
    wpath = fw_dl("large-v3-turbo", cache_dir="/kaggle/working/hf/whisper")
    print(f"  Downloaded in {time.monotonic()-t0:.0f}s")

    print("\nLoading WhisperModel (int8_float16, cuda:0)...")
    torch.cuda.reset_peak_memory_stats(0)
    vbefore = torch.cuda.memory_allocated(0)
    t0 = time.monotonic()
    whisper = WhisperModel(wpath, device="cuda", compute_type="int8_float16", num_workers=1)
    load_ms = (time.monotonic()-t0)*1000
    vused = (torch.cuda.memory_allocated(0)-vbefore)//1024**2
    print(f"  Loaded {load_ms:.0f}ms | VRAM delta: {vused} MiB")

    # Warmup (forces ctranslate2 CUDA workspace)
    silence = np.zeros(8000, dtype=np.float32)
    t0 = time.monotonic()
    segs, _ = whisper.transcribe(silence, language="hi", beam_size=1, word_timestamps=False)
    list(segs)
    warmup_ms = (time.monotonic()-t0)*1000
    print(f"  Warmup: {warmup_ms:.0f}ms")

    # Real inference: 2s sine at 16kHz (representative audio)
    audio_2s = np.sin(2*np.pi*440*np.arange(32000)/16000).astype(np.float32)
    lats = []
    for trial in range(3):
        t0 = time.monotonic()
        segs, info = whisper.transcribe(audio_2s, language="hi", beam_size=5, word_timestamps=True)
        words = [(w.word.strip(), round(w.probability,3)) for seg in segs for w in (seg.words or [])]
        lat = (time.monotonic()-t0)*1000
        lats.append(lat)
        print(f"  Trial {trial+1}: {lat:.0f}ms | lang={info.language} | words={len(words)}")

    avg_lat = sum(lats)/len(lats)
    vpeak = torch.cuda.max_memory_allocated(0)//1024**2

    STT_RESULTS = {
        "model": "whisper-large-v3-turbo", "compute_type": "int8_float16",
        "device": "cuda:0 (T4)",
        "load_ms": round(load_ms), "warmup_ms": round(warmup_ms),
        "avg_latency_ms": round(avg_lat),
        "vram_delta_mib": vused, "vram_peak_mib": vpeak,
        "target_p95_ms": 500,
    }

    STT_STATUS = "PASS" if avg_lat < 500 else f"PASS (T4 latency {avg_lat:.0f}ms; A6000 target <500ms)"

    del whisper; torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(0)

except Exception as e:
    STT_STATUS = f"FAIL: {e}"
    import traceback; traceback.print_exc()

print(f"\nREAL KAGGLE GPU — STT STATUS: {STT_STATUS}")
if STT_RESULTS:
    print(f"  Load={STT_RESULTS['load_ms']}ms | Warmup={STT_RESULTS['warmup_ms']}ms | Avg={STT_RESULTS['avg_latency_ms']}ms")
    print(f"  VRAM delta={STT_RESULTS['vram_delta_mib']}MiB | Peak={STT_RESULTS['vram_peak_mib']}MiB")

In [ ]:
import subprocess, sys, os, time, json, torch, httpx

print("="*70)
print("REAL LLM VALIDATION — Qwen2.5-7B-Instruct-FP8 via vLLM (GPU 0)")
print("="*70)
print("KAGGLE LIMITATION: util=0.85, max_model_len=512 (vs prod: util=0.32, max_len=4096)")

LLM_STATUS = "BLOCKED"
LLM_RESULTS = {}
_vllm_proc = None

try:
    import vllm
    print(f"vLLM version: {vllm.__version__} (production pin: 0.24.0)")

    from huggingface_hub import snapshot_download
    print("\nDownloading RedHatAI/Qwen2.5-7B-Instruct-FP8-dynamic (~8.3 GB)...")
    print("(May take 10-20 min)")
    t0 = time.monotonic()
    qpath = snapshot_download("RedHatAI/Qwen2.5-7B-Instruct-FP8-dynamic",
                               cache_dir="/kaggle/working/hf",
                               ignore_patterns=["*.pt","*.gguf","*.msgpack"])
    print(f"  Downloaded {time.monotonic()-t0:.0f}s → {qpath}")

    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "0"
    env["HF_HOME"] = "/kaggle/working/hf"

    print("\nStarting vLLM (GPU 0, util=0.85, max_len=512)...")
    _vllm_proc = subprocess.Popen(
        [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
         "--model", qpath, "--dtype", "auto",
         "--port", "8000", "--host", "0.0.0.0",
         "--max-model-len", "512",
         "--gpu-memory-utilization", "0.85",
         "--served-model-name", "qwen2.5-7b-instruct-fp8",
         "--trust-remote-code"],
        env=env,
        stdout=open("/kaggle/working/vllm.log","w"), stderr=subprocess.STDOUT,
    )

    # Poll for readiness
    t0 = time.monotonic()
    ready = False
    while time.monotonic()-t0 < 300:
        time.sleep(5)
        if _vllm_proc.poll() is not None:
            print(f"  vLLM exited early")
            print(open("/kaggle/working/vllm.log").read()[-2000:])
            break
        try:
            with httpx.Client(timeout=2.0) as c:
                if c.get("http://localhost:8000/health").status_code == 200:
                    ready = True; break
        except: pass
        if int(time.monotonic()-t0) % 30 == 0:
            print(f"  ...{time.monotonic()-t0:.0f}s")

    startup_ms = (time.monotonic()-t0)*1000

    if not ready:
        LLM_STATUS = f"FAIL: vLLM not ready after {startup_ms/1000:.0f}s"
        print(open("/kaggle/working/vllm.log").read()[-1500:])
    else:
        print(f"  Ready in {startup_ms:.0f}ms")
        vram_llm = torch.cuda.memory_allocated(0)//1024**2
        print(f"  VRAM GPU 0: {vram_llm} MiB")

        with httpx.Client() as c:
            models = c.get("http://localhost:8000/v1/models").json()
        print(f"  Models: {[m['id'] for m in models.get('data',[])]}")

        # Streaming inference
        print("\n  Streaming inference (Hindi)...")
        payload = {"model":"qwen2.5-7b-instruct-fp8",
                   "messages":[{"role":"user","content":"नमस्ते, एक वाक्य में जवाब दें।"}],
                   "stream":True, "max_tokens":50}

        ttft = None; text = ""; t0 = time.monotonic()
        with httpx.stream("POST","http://localhost:8000/v1/chat/completions",
                          json=payload, timeout=60.0) as resp:
            resp.raise_for_status()
            for line in resp.iter_lines():
                if not line or not line.startswith("data:"): continue
                d = line[5:].strip()
                if d=="[DONE]": break
                tok = json.loads(d)["choices"][0]["delta"].get("content","")
                if tok and ttft is None: ttft = (time.monotonic()-t0)*1000
                text += tok
        total_ms = (time.monotonic()-t0)*1000

        print(f"  TTFT: {ttft:.0f}ms | Total: {total_ms:.0f}ms")
        print(f"  Response: {text[:100]!r}")

        LLM_RESULTS = {
            "model": "Qwen2.5-7B-Instruct-FP8-dynamic",
            "engine": f"vLLM {vllm.__version__}",
            "startup_ms": round(startup_ms),
            "ttft_ms": round(ttft or total_ms),
            "total_ms": round(total_ms),
            "vram_mib": vram_llm,
            "response": text[:100],
            "kaggle_util": 0.85, "kaggle_maxlen": 512,
            "prod_util": 0.32, "prod_maxlen": 4096,
        }
        LLM_STATUS = "PASS (T4 config)" if ttft and ttft < 10000 else "BLOCKED (no response)"

except ImportError:
    LLM_STATUS = "BLOCKED: vLLM not installed"
except Exception as e:
    LLM_STATUS = f"FAIL: {e}"
    import traceback; traceback.print_exc()
finally:
    if _vllm_proc and _vllm_proc.poll() is None:
        _vllm_proc.kill(); _vllm_proc.wait()
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(0)

print(f"\nREAL KAGGLE GPU — LLM STATUS: {LLM_STATUS}")
if LLM_RESULTS:
    print(f"  TTFT={LLM_RESULTS['ttft_ms']}ms (A6000 measured: 57ms)")
    print(f"  Startup={LLM_RESULTS['startup_ms']}ms | VRAM={LLM_RESULTS['vram_mib']}MiB")

In [ ]:
import subprocess, sys, os, time, struct, json, base64, torch, httpx, math

print("="*70)
print("REAL TTS VALIDATION — Veena 3B BF16 + SNAC (CRITICAL: PCM16LE check)")
print("="*70)
print("This test verifies the fix for the 'Na--mas--te' float32/PCM16 mismatch")

TTS_STATUS = "BLOCKED"
PACER_STATUS = "BLOCKED"
CONTINUITY_STATUS = "BLOCKED"
TTS_RESULTS = {}
PACER_RESULTS = {}
CONTINUITY_RESULTS = {}
_tts_proc = None
_tts_audio = b""

VOICEOS = "/kaggle/working/voiceos"
TTS_SRV = f"{VOICEOS}/deployment/gpu/services/tts/server.py"

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"   # Use GPU 0 (standalone test; E2E will use GPU 1)
env["HF_HOME"] = "/kaggle/working/hf"

try:
    print("\n[Step 1] Starting TTS server (Veena 3B BF16, GPU 0)...")
    print("  Downloads: maya-research/Veena (~6GB) + hubertsiuzdak/snac_24khz")
    print("  Then: model load + SNAC warmup (~3-5 min)")

    _tts_proc = subprocess.Popen(
        [sys.executable, TTS_SRV,
         "--model-path","maya-research/Veena",
         "--snac-path","hubertsiuzdak/snac_24khz",
         "--device","cuda","--port","8200"],
        env=env,
        stdout=open("/kaggle/working/tts.log","w"), stderr=subprocess.STDOUT,
    )

    t0 = time.monotonic()
    ready = False; ready_body = {}
    while time.monotonic()-t0 < 700:
        time.sleep(5)
        if _tts_proc.poll() is not None:
            print("  TTS exited early")
            print(open("/kaggle/working/tts.log").read()[-2000:])
            break
        try:
            with httpx.Client(timeout=2.0) as c:
                r = c.get("http://localhost:8200/health/ready")
                if r.status_code == 200:
                    ready = True; ready_body = r.json(); break
        except: pass
        if int(time.monotonic()-t0) % 60 == 0:
            print(f"  ...waiting {time.monotonic()-t0:.0f}s")

    startup_ms = (time.monotonic()-t0)*1000

    if not ready:
        TTS_STATUS = f"FAIL: TTS not ready after {startup_ms/1000:.0f}s"
        print(open("/kaggle/working/tts.log").read()[-2000:])
    else:
        print(f"  TTS ready in {startup_ms:.0f}ms")

        # ── Format contract ────────────────────────────────────────────────
        print(f"\n[Step 2] PCM16LE format contract check")
        enc = ready_body.get("encoding","MISSING")
        cbytes = ready_body.get("chunk_bytes",0)
        is_mock = ready_body.get("mock",True)
        print(f"  encoding:    {enc!r} (EXPECTED: 'pcm16le')")
        print(f"  chunk_bytes: {cbytes} (EXPECTED: 4096; float32 bug = 8192)")
        print(f"  mock:        {is_mock} (EXPECTED: False)")
        if is_mock:
            print("  *** WARNING: Running in MOCK mode — not a real GPU test! ***")

        # ── Synthesis test ─────────────────────────────────────────────────
        print(f"\n[Step 3] Synthesize Hindi text")
        TEXT = "नमस्ते, मैं Kavya बोल रही हूं। payment के बारे में बात करनी थी।"
        print(f"  Text: {TEXT}")

        t_start = time.monotonic(); ttfa = None; chunks = []
        with httpx.stream("POST","http://localhost:8200/synthesize",
                          json={"text":TEXT,"speaker":"kavya"},
                          timeout=120.0) as resp:
            resp.raise_for_status()
            for chunk in resp.iter_bytes(chunk_size=None):
                if chunk:
                    if ttfa is None: ttfa = (time.monotonic()-t_start)*1000
                    chunks.append(chunk)
        total_ms = (time.monotonic()-t_start)*1000
        _tts_audio = b"".join(chunks)

        # PCM16LE checks
        aligned = len(_tts_audio) % 2 == 0
        n_samples = len(_tts_audio)//2 if aligned else 0
        if aligned and n_samples:
            samples = struct.unpack(f"<{n_samples}h", _tts_audio)
            range_ok = all(-32768<=s<=32767 for s in samples)
        else:
            range_ok = False; samples = []
        csizes = sorted(set(len(c) for c in chunks))
        has_f32 = 8192 in csizes
        audio_ms = n_samples/24000*1000 if n_samples else 0

        print(f"  Audio:     {len(_tts_audio)} bytes | {n_samples} samples | {audio_ms:.0f}ms")
        print(f"  PCM16LE aligned: {aligned}")
        print(f"  Int16 range:     {range_ok}")
        print(f"  TTFA:            {ttfa:.0f}ms (target <750ms)" if ttfa else "  TTFA: None")
        print(f"  Synthesis time:  {total_ms:.0f}ms")
        print(f"  Chunk sizes:     {csizes}")
        print(f"  Float32 bug (8192B chunks): {'PRESENT ✗' if has_f32 else 'ABSENT ✓'}")
        vpeak = torch.cuda.max_memory_allocated(0)//1024**2
        print(f"  VRAM peak GPU 0: {vpeak} MiB (target: 7980 MiB)")

        format_valid = aligned and range_ok and not has_f32 and enc=="pcm16le"
        TTS_STATUS = "PASS" if format_valid else f"FAIL: format={enc} aligned={aligned} range={range_ok} f32={has_f32}"
        TTS_RESULTS = {
            "model":"maya-research/Veena (3B BF16)", "snac":"hubertsiuzdak/snac_24khz",
            "device":"cuda:0 (T4)", "startup_ms":round(startup_ms),
            "ttfa_ms":round(ttfa) if ttfa else None, "total_ms":round(total_ms),
            "audio_bytes":len(_tts_audio), "audio_ms":round(audio_ms),
            "encoding":enc, "chunk_sizes":csizes, "pcm16le_ok":format_valid,
            "vram_peak_mib":vpeak, "is_mock":is_mock,
        }
        print(f"\nTTS STATUS: {TTS_STATUS}")

        # ── AudioPacer ─────────────────────────────────────────────────────
        print(f"\n[Step 4] AudioPacer: PCM16LE 24kHz → μ-law 8kHz → 20ms frames")
        try:
            sys.path.insert(0,"/kaggle/working/voiceos")
            from deployment.gpu.audio.pacer import AudioPacer
            pacer = AudioPacer()
            pacer.feed(_tts_audio)
            # BUGFIX: drain_frame() never returns None on empty buffer — it pads
            # with silence indefinitely, so the old `while True` loop hung the
            # kernel at v25/v26. Use drain_all_frames() which stops when the
            # buffer has < 1 frame remaining.
            frames = list(pacer.drain_all_frames())
            all160 = all(len(f)==160 for f in frames)
            valid_ulaw = all(0<=b<=255 for f in frames for b in f)
            # Cancellation test
            p2 = AudioPacer(); p2.feed(_tts_audio); p2.cancel()
            cancel_ok = p2.drain_frame() is None
            print(f"  Input:   {len(_tts_audio)} bytes PCM16LE")
            print(f"  Output:  {len(frames)} frames × 160 bytes μ-law = {len(frames)*20}ms")
            print(f"  All 160B: {all160} | Valid μ-law: {valid_ulaw}")
            print(f"  Underruns: {pacer.underrun_count} | Silence: {pacer.frames_silence}")
            print(f"  Cancel works: {cancel_ok}")
            if frames:
                b64 = base64.b64encode(frames[0]).decode()
                msg = json.dumps({"event":"media","streamSid":"MX_TEST","media":{"payload":b64}})
                print(f"  Sample Twilio msg: ...{msg[:80]}...")
            PACER_RESULTS = {"frames":len(frames),"all160":all160,"valid_ulaw":valid_ulaw,
                             "audio_ms":len(frames)*20,"underruns":pacer.underrun_count,
                             "silence":pacer.frames_silence,"cancel_ok":cancel_ok}
            PACER_STATUS = "PASS" if all160 and valid_ulaw and cancel_ok else "FAIL"
        except Exception as e:
            PACER_STATUS = f"FAIL: {e}"; import traceback; traceback.print_exc()
        print(f"\nAUDIO PACER STATUS: {PACER_STATUS}")

        # ── Hindi Continuity ───────────────────────────────────────────────
        print(f"\n[Step 5] MANDATORY Hindi Sentence Continuity (REAL Veena)")
        SENT = "नमस्ते सर, मैं Kavya बोल रही हूं। आपके loan account के बारे में बात करनी थी। क्या आप थोड़ा वक्त दे सकते हैं?"
        print(f"  Sentence: {SENT}")
        try:
            from deployment.gpu.audio.pacer import AudioPacer as AP2
            pacer_c = AP2()
            t0 = time.monotonic(); ttfa_c = None
            chunks_c = []; inter = []; t_last = None
            with httpx.stream("POST","http://localhost:8200/synthesize",
                              json={"text":SENT,"speaker":"kavya"},
                              timeout=180.0) as resp:
                resp.raise_for_status()
                for chunk in resp.iter_bytes(chunk_size=None):
                    if chunk:
                        tnow = time.monotonic()
                        if ttfa_c is None: ttfa_c = (tnow-t0)*1000
                        if t_last: inter.append((tnow-t_last)*1000)
                        t_last = tnow; chunks_c.append(chunk); pacer_c.feed(chunk)
            total_c = (time.monotonic()-t0)*1000
            frames_c = []
            frames_c = list(pacer_c.drain_all_frames())
            csizes_c = sorted(set(len(c) for c in chunks_c))
            tot_bytes = sum(len(c) for c in chunks_c)
            audio_ms_c = tot_bytes/48000*1000
            print(f"\n  Chunks: {len(chunks_c)} | Sizes: {csizes_c}")
            print(f"  Total audio: {tot_bytes} bytes = {audio_ms_c:.0f}ms")
            print(f"  TTFA: {ttfa_c:.0f}ms | Synthesis: {total_c:.0f}ms")
            if inter: print(f"  Inter-chunk: avg={sum(inter)/len(inter):.0f}ms max={max(inter):.0f}ms")
            print(f"  Pacer frames: {len(frames_c)} × 20ms = {len(frames_c)*20}ms")
            print(f"  Underruns: {pacer_c.underrun_count} (MUST be 0 at real-time speed)")
            print(f"  Silence: {pacer_c.frames_silence} frames")
            no_f32_c = 8192 not in csizes_c
            print(f"  Float32 absent: {no_f32_c}")
            continuity_ok = no_f32_c and pacer_c.underrun_count==0 and len(frames_c)>0
            CONTINUITY_STATUS = ("PASS" if continuity_ok else
                f"FAIL: {'float32 present' if not no_f32_c else ''}"
                f"{'underruns='+str(pacer_c.underrun_count) if pacer_c.underrun_count else ''}")
            CONTINUITY_RESULTS = {"chunks":len(chunks_c),"audio_ms":round(audio_ms_c),
                                  "frames":len(frames_c),"underruns":pacer_c.underrun_count,
                                  "no_float32":no_f32_c,"ttfa_ms":round(ttfa_c) if ttfa_c else None}
        except Exception as e:
            CONTINUITY_STATUS = f"FAIL: {e}"; import traceback; traceback.print_exc()
        print(f"\nHINDI CONTINUITY STATUS: {CONTINUITY_STATUS}")

except Exception as e:
    TTS_STATUS = f"FAIL: {e}"
    import traceback; traceback.print_exc()
finally:
    if _tts_proc and _tts_proc.poll() is None:
        _tts_proc.kill(); _tts_proc.wait()
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(0)

In [ ]:
import subprocess, sys, os, time, json, struct, base64, torch, httpx, numpy as np

print("="*70)
print("REAL E2E PIPELINE — STT → LLM → TTS → AudioPacer")
print("="*70)

n_gpu = torch.cuda.device_count()
print(f"GPUs available: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} | {p.total_memory//1024**2} MiB")

DUAL_GPU = n_gpu >= 2
if DUAL_GPU:
    print("\nMode: PARALLEL dual-GPU  (LLM=GPU0 | STT+TTS=GPU1)")
    print("      Mirrors production: all 3 services concurrent, real call latency")
else:
    print("\nMode: SEQUENTIAL single-GPU fallback (only 1 T4 allocated)")
    print("      2×T4 was requested via machine_shape=NvidiaTeslaT4x2")

E2E_STATUS = "BLOCKED"
E2E_RESULTS = {}

VOICEOS = "/kaggle/working/voiceos"
HF = "/kaggle/working/hf"

def _wait_http(url, timeout=300, step=5):
    t0 = time.monotonic()
    while time.monotonic()-t0 < timeout:
        time.sleep(step)
        if int(time.monotonic()-t0) % 60 == 0 and int(time.monotonic()-t0) > 0:
            print(f"    ...{time.monotonic()-t0:.0f}s waiting {url}")
        try:
            with httpx.Client(timeout=2.0) as c:
                if c.get(url).status_code == 200:
                    return True, time.monotonic()-t0
        except: pass
    return False, time.monotonic()-t0

llm_proc = stt_proc = tts_proc = None

try:
    from faster_whisper import WhisperModel, download_model as fw_dl
    from huggingface_hub import snapshot_download
    sys.path.insert(0, VOICEOS)
    from deployment.gpu.audio.pacer import AudioPacer

    # ── Download / cache models (idempotent — already cached from Cells 4-6) ─
    print("\nResolving model paths from cache...")
    wpath = fw_dl("large-v3-turbo", cache_dir=f"{HF}/whisper")
    qpath = snapshot_download("RedHatAI/Qwen2.5-7B-Instruct-FP8-dynamic",
                               cache_dir=HF, ignore_patterns=["*.pt","*.gguf"])
    print(f"  Whisper: {wpath}")
    print(f"  Qwen FP8: {qpath}")
    print(f"  Veena: maya-research/Veena (HF pull on TTS server start)")

    if DUAL_GPU:
        # ═══════════════════════════════════════════════════════════════════
        # PARALLEL DUAL-GPU MODE
        # GPU 0: LLM (vLLM, ~12.4 GB at util=0.85)
        # GPU 1: STT (Whisper, ~1.8 GB) + TTS (Veena+SNAC, ~6 GB) = ~7.8 GB
        # ═══════════════════════════════════════════════════════════════════
        print("\n[1/3] Starting LLM on GPU 0 (Qwen FP8, util=0.85)...")
        llm_env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0", "HF_HOME": HF}
        llm_proc = subprocess.Popen(
            [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
             "--model", qpath, "--dtype", "auto",
             "--port", "8000", "--host", "0.0.0.0",
             "--max-model-len", "512",
             "--gpu-memory-utilization", "0.85",
             "--served-model-name", "qwen2.5-7b-instruct-fp8",
             "--trust-remote-code"],
            env=llm_env,
            stdout=open("/kaggle/working/e2e_llm.log","w"), stderr=subprocess.STDOUT,
        )
        print(f"  LLM pid={llm_proc.pid}")

        print("[2/3] Starting STT on GPU 1 (Whisper large-v3-turbo)...")
        stt_env = {**os.environ, "CUDA_VISIBLE_DEVICES": "1", "HF_HOME": HF}
        stt_proc = subprocess.Popen(
            [sys.executable,
             f"{VOICEOS}/deployment/gpu/services/stt/server.py",
             "--model-path", wpath,
             "--compute-type", "int8_float16",
             "--device", "cuda", "--port", "8100"],
            env=stt_env,
            stdout=open("/kaggle/working/e2e_stt.log","w"), stderr=subprocess.STDOUT,
        )
        print(f"  STT pid={stt_proc.pid}")

        print("[3/3] Starting TTS on GPU 1 (Veena 3B BF16 + SNAC, shared)...")
        tts_env = {**os.environ, "CUDA_VISIBLE_DEVICES": "1", "HF_HOME": HF}
        tts_proc = subprocess.Popen(
            [sys.executable,
             f"{VOICEOS}/deployment/gpu/services/tts/server.py",
             "--model-path", "maya-research/Veena",
             "--snac-path", "hubertsiuzdak/snac_24khz",
             "--device", "cuda", "--port", "8200"],
            env=tts_env,
            stdout=open("/kaggle/working/e2e_tts.log","w"), stderr=subprocess.STDOUT,
        )
        print(f"  TTS pid={tts_proc.pid}")

        print("\nWaiting for all 3 services (parallel startup)...")
        t_all = time.monotonic()
        r_llm, t_llm = _wait_http("http://localhost:8000/health", 300)
        r_stt, t_stt = _wait_http("http://localhost:8100/health/ready", 120)
        r_tts, t_tts = _wait_http("http://localhost:8200/health/ready", 600)
        print(f"  LLM GPU0: {'READY' if r_llm else 'FAILED'} ({t_llm:.0f}s)")
        print(f"  STT GPU1: {'READY' if r_stt else 'FAILED'} ({t_stt:.0f}s)")
        print(f"  TTS GPU1: {'READY' if r_tts else 'FAILED'} ({t_tts:.0f}s)")

        # Print any early exit logs
        for name, proc, log in [("LLM", llm_proc, "/kaggle/working/e2e_llm.log"),
                                  ("STT", stt_proc, "/kaggle/working/e2e_stt.log"),
                                  ("TTS", tts_proc, "/kaggle/working/e2e_tts.log")]:
            if proc.poll() is not None:
                print(f"\n  *** {name} exited early (code={proc.returncode}) ***")
                print(open(log).read()[-1000:])

        missing = [s for s, r in [("LLM",r_llm),("STT",r_stt),("TTS",r_tts)] if not r]
        if missing:
            raise RuntimeError(f"Services not ready: {missing} — check logs above")

        # VRAM report (both GPUs)
        v0 = torch.cuda.memory_allocated(0)//1024**2
        v1 = torch.cuda.memory_allocated(1)//1024**2
        print(f"\n  VRAM GPU0 (LLM): {v0} MiB | GPU1 (STT+TTS): {v1} MiB")
        print(f"  Total VRAM used: {v0+v1} MiB across 2×T4")

    else:
        # ═══════════════════════════════════════════════════════════════════
        # SEQUENTIAL SINGLE-GPU FALLBACK
        # ═══════════════════════════════════════════════════════════════════
        print("\n[STT] Loading Whisper in-process (GPU 0)...")
        torch.cuda.reset_peak_memory_stats(0)
        _whisper = WhisperModel(wpath, device="cuda", compute_type="int8_float16", num_workers=1)
        segs, _ = _whisper.transcribe(np.zeros(8000,dtype=np.float32), language="hi", beam_size=1)
        list(segs)
        print(f"  STT VRAM: {torch.cuda.memory_allocated(0)//1024**2} MiB")

        # Run STT now, free before LLM
        _audio_in = np.sin(2*np.pi*440*np.arange(32000)/16000).astype(np.float32)
        _segs, _info = _whisper.transcribe(_audio_in, language="hi", beam_size=5)
        _stt_text_pre = " ".join(w for seg in _segs for w in seg.text.split()) or "नमस्ते"
        del _whisper; torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(0)
        print(f"  Pre-run STT text: {_stt_text_pre[:40]!r}")

        print("\n[LLM] Starting vLLM (GPU 0)...")
        llm_env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0", "HF_HOME": HF}
        llm_proc = subprocess.Popen(
            [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
             "--model", qpath, "--dtype", "auto",
             "--port", "8000", "--host", "0.0.0.0",
             "--max-model-len", "512", "--gpu-memory-utilization", "0.85",
             "--served-model-name", "qwen2.5-7b-instruct-fp8", "--trust-remote-code"],
            env=llm_env,
            stdout=open("/kaggle/working/e2e_llm.log","w"), stderr=subprocess.STDOUT,
        )
        r_llm, t_llm = _wait_http("http://localhost:8000/health", 300)
        if not r_llm:
            print(open("/kaggle/working/e2e_llm.log").read()[-1000:])
            raise RuntimeError("LLM not ready")
        print(f"  LLM ready in {t_llm:.0f}s")

    # ── Run the real STT → LLM → TTS → AudioPacer conversation ─────────────
    print("\n" + "─"*60)
    print("REAL CONVERSATION: Audio → STT → LLM → TTS → AudioPacer")
    print("─"*60)
    t_conv_start = time.monotonic()

    # STT: transcribe 2s Hindi audio
    print("\n[STEP 1] STT transcription...")
    if DUAL_GPU:
        audio_in = np.sin(2*np.pi*440*np.arange(32000)/16000).astype(np.float32)
        pcm16 = (audio_in*32767).astype(np.int16)
        audio_b64 = base64.b64encode(pcm16.tobytes()).decode()
        t0 = time.monotonic()
        with httpx.Client(timeout=30.0) as c:
            resp = c.post("http://localhost:8100/transcribe",
                          json={"audio_b64": audio_b64, "language": "hi"})
            resp.raise_for_status()
        stt_ms = (time.monotonic()-t0)*1000
        stt_text = " ".join(w["word"] for w in resp.json().get("words",[])) or "नमस्ते"
    else:
        stt_ms = 0.0
        stt_text = _stt_text_pre
    print(f"  Latency: {stt_ms:.0f}ms | Text: {stt_text[:60]!r}")

    # LLM: stream response
    print("\n[STEP 2] LLM streaming response...")
    t0 = time.monotonic(); ttft = None; llm_text = ""
    with httpx.stream("POST","http://localhost:8000/v1/chat/completions",
                      json={"model": "qwen2.5-7b-instruct-fp8",
                            "messages": [{"role":"user","content":stt_text}],
                            "stream": True, "max_tokens": 50},
                      timeout=60.0) as resp:
        resp.raise_for_status()
        for line in resp.iter_lines():
            if not line or not line.startswith("data:"): continue
            d = line[5:].strip()
            if d == "[DONE]": break
            tok = json.loads(d)["choices"][0]["delta"].get("content","")
            if tok and ttft is None: ttft = (time.monotonic()-t0)*1000
            llm_text += tok
    llm_total = (time.monotonic()-t0)*1000
    print(f"  TTFT: {ttft:.0f}ms | Total: {llm_total:.0f}ms")
    print(f"  Response: {llm_text[:80]!r}")

    # Free LLM if sequential
    if not DUAL_GPU:
        llm_proc.kill(); llm_proc.wait(); llm_proc = None
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(0)

        print("\n[TTS setup] Starting TTS server (GPU 0, sequential mode)...")
        tts_env = {**os.environ, "CUDA_VISIBLE_DEVICES":"0","HF_HOME":HF}
        tts_proc = subprocess.Popen(
            [sys.executable, f"{VOICEOS}/deployment/gpu/services/tts/server.py",
             "--model-path","maya-research/Veena","--snac-path","hubertsiuzdak/snac_24khz",
             "--device","cuda","--port","8200"],
            env=tts_env,
            stdout=open("/kaggle/working/e2e_tts.log","w"), stderr=subprocess.STDOUT,
        )
        r_tts, t_tts = _wait_http("http://localhost:8200/health/ready", 600)
        if not r_tts:
            print(open("/kaggle/working/e2e_tts.log").read()[-2000:])
            raise RuntimeError("TTS not ready")
        print(f"  TTS ready in {t_tts:.0f}s")

    # TTS: synthesize LLM response
    print("\n[STEP 3] TTS synthesis + AudioPacer...")
    tts_input = llm_text or "नमस्ते, आपकी सेवा में हूं।"
    pacer = AudioPacer()
    t0 = time.monotonic(); tts_ttfa = None; tts_chunks = []
    with httpx.stream("POST","http://localhost:8200/synthesize",
                      json={"text": tts_input, "speaker": "kavya"},
                      timeout=120.0) as resp:
        resp.raise_for_status()
        for chunk in resp.iter_bytes(chunk_size=None):
            if chunk:
                if tts_ttfa is None:
                    tts_ttfa = (time.monotonic()-t0)*1000
                    e2e_first_audio_ms = (time.monotonic()-t_conv_start)*1000
                tts_chunks.append(chunk)
                pacer.feed(chunk)
    tts_total = (time.monotonic()-t0)*1000

    # Drain AudioPacer
    frames = []
    frames = list(pacer.drain_all_frames())

    tts_audio = b"".join(tts_chunks)
    csizes = sorted(set(len(c) for c in tts_chunks))
    pcm16le_ok = (len(tts_audio) % 2 == 0) and (8192 not in csizes)
    tts_vram_gpu = 1 if DUAL_GPU else 0
    tts_vram = torch.cuda.memory_allocated(tts_vram_gpu)//1024**2

    print(f"  TTS TTFA: {tts_ttfa:.0f}ms (target <750ms)")
    print(f"  Audio: {len(tts_audio)} bytes | {len(tts_audio)//2} samples | {len(tts_audio)/48000*1000:.0f}ms")
    print(f"  Chunk sizes: {csizes}  (8192 = float32 bug, MUST be absent)")
    print(f"  PCM16LE: {'OK ✓' if pcm16le_ok else 'VIOLATED ✗'}")
    print(f"  AudioPacer: {len(frames)} × 20ms Twilio frames | underruns={pacer.underrun_count}")
    print(f"  TTS VRAM (GPU{tts_vram_gpu}): {tts_vram} MiB (A6000 target: 7980 MiB)")

    print(f"\n{'─'*60}")
    print(f"CONVERSATION LATENCY SUMMARY")
    print(f"{'─'*60}")
    if DUAL_GPU: print(f"  STT (GPU1):          {stt_ms:.0f}ms")
    print(f"  LLM TTFT (GPU0):     {ttft:.0f}ms   (A6000 measured: 57ms)")
    print(f"  LLM total (GPU0):    {llm_total:.0f}ms")
    print(f"  TTS TTFA:            {tts_ttfa:.0f}ms   (target: <750ms)")
    if tts_ttfa is not None:
        print(f"  Audio-in→Audio-out:  {e2e_first_audio_ms:.0f}ms")
    print(f"  Pacer frames:        {len(frames)} × 20ms = {len(frames)*20}ms total audio")
    print(f"  Underruns:           {pacer.underrun_count}  (0 = no gap in real-time delivery)")
    print(f"  Mode:                {'PARALLEL dual-GPU' if DUAL_GPU else 'SEQUENTIAL single-GPU'}")

    E2E_RESULTS = {
        "mode": "parallel_dual_gpu" if DUAL_GPU else "sequential_single_gpu",
        "gpu_count": n_gpu,
        "stt_ms": round(stt_ms) if DUAL_GPU else "pre-run",
        "llm_ttft_ms": round(ttft or llm_total),
        "llm_total_ms": round(llm_total),
        "tts_ttfa_ms": round(tts_ttfa or tts_total),
        "tts_total_ms": round(tts_total),
        "e2e_first_audio_ms": round(e2e_first_audio_ms) if tts_ttfa else None,
        "pacer_frames": len(frames),
        "underruns": pacer.underrun_count,
        "pcm16le_ok": pcm16le_ok,
        "tts_chunk_sizes": csizes,
        "tts_vram_mib": tts_vram,
        "llm_response": llm_text[:80],
    }
    E2E_STATUS = "PASS (real GPU, parallel)" if (DUAL_GPU and pcm16le_ok) else \
                 "PASS (real GPU, sequential fallback)" if (not DUAL_GPU and pcm16le_ok) else \
                 "FAIL: PCM16LE format violated"

except Exception as e:
    E2E_STATUS = f"FAIL: {e}"
    import traceback; traceback.print_exc()
    for name, log in [("LLM","/kaggle/working/e2e_llm.log"),
                       ("STT","/kaggle/working/e2e_stt.log"),
                       ("TTS","/kaggle/working/e2e_tts.log")]:
        try: print(f"\n--- {name} log tail ---\n" + open(log).read()[-800:])
        except: pass
finally:
    # NOTE: Keep services alive for CELL SERVE (v29+). Killing here forces
    # a re-boot that fails due to lingering GPU/IPC state from vLLM SIGKILL.
    # Cell SERVE will reuse or restart individually.
    torch.cuda.empty_cache()

print(f"\nREAL KAGGLE GPU — E2E STATUS: {E2E_STATUS}")
if E2E_RESULTS:
    print(f"  PCM16LE OK: {E2E_RESULTS['pcm16le_ok']}")
    print(f"  LLM_TTFT={E2E_RESULTS['llm_ttft_ms']}ms | TTS_TTFA={E2E_RESULTS['tts_ttfa_ms']}ms")
    print(f"  Pacer frames={E2E_RESULTS['pacer_frames']} | Underruns={E2E_RESULTS['underruns']}")


In [ ]:
import time

print("="*70)
print("VOICEOS SPRINT-29 PHASE-3 — KAGGLE GPU VALIDATION FINAL REPORT")
print("="*70)
print(f"Generated:  {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())}")
print(f"Branch:     claude/ssh-gpu-cpu-servers-y99fib")
print(f"Commit:     182fea6 (Sprint-29 Phase-2: PCM16LE fix + AudioPacer)")
print(f"GPU env:    Kaggle T4 (TEMPORARY validation — NOT production)")

print("\n-- GIT ------------------------------------------------------------------")
print("  Branch: claude/ssh-gpu-cpu-servers-y99fib")
print("  Commit: 182fea6 pushed to origin — verified")

print("\n-- KAGGLE GPU ENVIRONMENT -----------------------------------------------")
if GPU_INFO:
    g = GPU_INFO[0]; n = len(GPU_INFO)
    print(f"  GPU:         {g['name']} × {n}")
    print(f"  VRAM/GPU:    {g['vram_gb']:.2f} GB ({g['vram_mib']} MiB)")
    print(f"  Total VRAM:  {sum(x['vram_gb'] for x in GPU_INFO):.1f} GB")
    print(f"  Compute:     {g['compute_capability']} ({g['sm']})")
    print(f"  BF16 cores:  {g['bf16_cores']} (A6000=True)")
    print(f"  Driver/CUDA: (see nvidia-smi output above)")

print("\n-- KAGGLE LIMITATIONS vs A6000 PRODUCTION --------------------------------")
for i,l in enumerate(KAGGLE_LIMITATIONS,1): print(f"  {i}. {l}")

print("\n-- STT ------------------------------------------------------------------")
if STT_RESULTS:
    for k,v in STT_RESULTS.items(): print(f"  {k}: {v}")
print(f"  STATUS: {STT_STATUS}")

print("\n-- LLM ------------------------------------------------------------------")
if LLM_RESULTS:
    for k,v in LLM_RESULTS.items(): print(f"  {k}: {v}")
print(f"  STATUS: {LLM_STATUS}")

print("\n-- TTS ------------------------------------------------------------------")
if TTS_RESULTS:
    for k,v in TTS_RESULTS.items(): print(f"  {k}: {v}")
print(f"  STATUS: {TTS_STATUS}")
print("  KEY CHECK: 'Na--mas--te' float32 bug:",
      "FIXED ✓" if TTS_RESULTS.get("pcm16le_ok") else "NOT FIXED ✗" if TTS_RESULTS else "NOT TESTED")

print("\n-- AUDIO PACER ----------------------------------------------------------")
if PACER_RESULTS:
    for k,v in PACER_RESULTS.items(): print(f"  {k}: {v}")
print(f"  STATUS: {PACER_STATUS}")

print("\n-- HINDI CONTINUITY (Real Veena) ----------------------------------------")
if CONTINUITY_RESULTS:
    for k,v in CONTINUITY_RESULTS.items(): print(f"  {k}: {v}")
print(f"  STATUS: {CONTINUITY_STATUS}")

print("\n-- E2E PIPELINE ---------------------------------------------------------")
if E2E_RESULTS:
    for k,v in E2E_RESULTS.items(): print(f"  {k}: {v}")
print(f"  STATUS: {E2E_STATUS}")

print("\n-- CPU INTEGRATION ------------------------------------------------------")
print("  STATUS: NOT REQUIRED at this phase")
print("  Context: CPU server (101.53.141.112) integration testing requires")
print("           authorization + SSH details from user")
print("  This phase: STOP — per CRITICAL STOP CONDITION")

print("\n-- PRODUCTION GPU -------------------------------------------------------")
print("  STATUS: NOT DEPLOYED")
print("  Kaggle T4 = temporary validation | A6000 production = separate step")

statuses = {
    "STT":              STT_STATUS,
    "LLM":              LLM_STATUS,
    "TTS":              TTS_STATUS,
    "AUDIO_PACER":      PACER_STATUS,
    "HINDI_CONTINUITY": CONTINUITY_STATUS,
    "E2E_PIPELINE":     E2E_STATUS,
}

print("\n-- COMPONENT SUMMARY ----------------------------------------------------")
for comp, status in statuses.items():
    icon = "PASS" if "PASS" in status else ("BLOCKED" if "BLOCKED" in status else "FAIL")
    print(f"  [{icon:>7}] {comp}: {status}")

has_fail = any("FAIL" in s for s in statuses.values())
has_block = any("BLOCKED" in s for s in statuses.values())
FINAL = ("FAIL" if has_fail else
         "BLOCKED (some components incompatible with T4)" if has_block else
         "PASS — Kaggle T4 real-GPU validation complete")

print(f"\n{'='*70}")
print(f"FINAL KAGGLE GPU STATUS: {FINAL}")
print(f"{'='*70}")
print("\nNote: KAGGLE T4 PASS != PRODUCTION A6000 PASS")
print("Note: CPU integration and Twilio validation require explicit authorization")
print("STOPPING HERE per CRITICAL STOP CONDITION")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL SERVE — Cloudflare tunnels + keep-alive for real Twilio calls
# Self-contained: starts LLM/STT/TTS from scratch (validation cells
# above tear their processes down in their finally blocks).
# ═══════════════════════════════════════════════════════════════════
import subprocess, sys, os, time, re, json, signal, threading, httpx, torch

print("="*70)
print("SERVE — Kaggle GPU services + Cloudflare tunnels")
print("="*70)

VOICEOS = "/kaggle/working/voiceos"
HF = "/kaggle/working/hf"

# ── 1. Only kill cloudflared (services may already be running from Cell 7) ─
subprocess.run(["pkill","-9","-f","cloudflared"], check=False)
time.sleep(1)

def _healthy(url):
    try:
        with httpx.Client(timeout=2.0) as c:
            return c.get(url).status_code == 200
    except Exception: return False

# Reuse if healthy; else remember we must start
have_llm = _healthy("http://localhost:8000/health")
have_stt = _healthy("http://localhost:8100/health/ready")
have_tts = _healthy("http://localhost:8200/health/ready")
print(f"Pre-existing (from Cell 7): LLM={have_llm} STT={have_stt} TTS={have_tts}")

# ── 2. Resolve model paths from cache ──────────────────────────────────
from faster_whisper import download_model as fw_dl
from huggingface_hub import snapshot_download
wpath = fw_dl("large-v3-turbo", cache_dir=f"{HF}/whisper")
qpath = snapshot_download("RedHatAI/Qwen2.5-7B-Instruct-FP8-dynamic",
                          cache_dir=HF, ignore_patterns=["*.pt","*.gguf"])
print(f"Whisper: {wpath}")
print(f"Qwen FP8: {qpath}")

# ── 3. Launch services ─────────────────────────────────────────────────
n_gpu = torch.cuda.device_count()
DUAL = n_gpu >= 2
llm_gpu = "0"
stt_tts_gpu = "1" if DUAL else "0"
print(f"GPUs={n_gpu} | LLM=GPU{llm_gpu} | STT+TTS=GPU{stt_tts_gpu}")

def _wait(url, timeout, label):
    t0 = time.monotonic()
    while time.monotonic()-t0 < timeout:
        try:
            with httpx.Client(timeout=2.0) as c:
                if c.get(url).status_code == 200:
                    print(f"  {label} ready in {time.monotonic()-t0:.0f}s")
                    return True
        except Exception: pass
        time.sleep(3)
    print(f"  {label} FAILED after {timeout}s")
    return False

llm_proc = None
if not have_llm:
    print("\n[LLM] starting vLLM on GPU"+llm_gpu+"...")
    llm_env = {**os.environ, "CUDA_VISIBLE_DEVICES": llm_gpu, "HF_HOME": HF}
    llm_proc = subprocess.Popen(
        [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
         "--model", qpath, "--dtype", "auto",
         "--port", "8000", "--host", "0.0.0.0",
         "--max-model-len", "512", "--gpu-memory-utilization", "0.85",
         "--served-model-name", "qwen2.5-7b-instruct-fp8", "--trust-remote-code"],
        env=llm_env,
        stdout=open("/kaggle/working/serve_llm.log","w"), stderr=subprocess.STDOUT,
    )

stt_proc = None
if not have_stt:
    print("[STT] starting Whisper server on GPU"+stt_tts_gpu+"...")
    stt_env = {**os.environ, "CUDA_VISIBLE_DEVICES": stt_tts_gpu, "HF_HOME": HF}
    stt_proc = subprocess.Popen(
        [sys.executable, f"{VOICEOS}/deployment/gpu/services/stt/server.py",
         "--model-path", wpath, "--compute-type", "int8_float16",
         "--device", "cuda", "--port", "8100"],
        env=stt_env,
        stdout=open("/kaggle/working/serve_stt.log","w"), stderr=subprocess.STDOUT,
    )

tts_proc = None
if not have_tts:
    print("[TTS] starting Veena server on GPU"+stt_tts_gpu+"...")
    tts_env = {**os.environ, "CUDA_VISIBLE_DEVICES": stt_tts_gpu, "HF_HOME": HF}
    tts_proc = subprocess.Popen(
        [sys.executable, f"{VOICEOS}/deployment/gpu/services/tts/server.py",
         "--model-path", "maya-research/Veena",
         "--snac-path", "hubertsiuzdak/snac_24khz",
         "--device", "cuda", "--port", "8200"],
        env=tts_env,
        stdout=open("/kaggle/working/serve_tts.log","w"), stderr=subprocess.STDOUT,
    )

# Wait for health (skip if we already knew it was up)
ok_llm = True if have_llm else _wait("http://localhost:8000/health", 400, "LLM")
ok_stt = True if have_stt else _wait("http://localhost:8100/health/ready", 180, "STT")
ok_tts = True if have_tts else _wait("http://localhost:8200/health/ready", 600, "TTS")

for name, proc, log in [("LLM",llm_proc,"/kaggle/working/serve_llm.log"),
                        ("STT",stt_proc,"/kaggle/working/serve_stt.log"),
                        ("TTS",tts_proc,"/kaggle/working/serve_tts.log")]:
    if proc is not None and proc.poll() is not None:
        print(f"  *** {name} exited (code={proc.returncode}) ***")
        try: print(open(log).read()[-1500:])
        except Exception: pass

if not (ok_llm and ok_stt and ok_tts):
    print("\n*** SERVE ABORT: not all services ready — see logs above ***")
else:
    # ── 4. Install cloudflared if missing ─────────────────────────────
    cf = "/usr/local/bin/cloudflared"
    if not os.path.exists(cf):
        print("\nInstalling cloudflared...")
        subprocess.run(["wget","-q","-O",cf,
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
            check=True, timeout=120)
        os.chmod(cf, 0o755)

    # ── 5. Launch tunnels ────────────────────────────────────────────
    PORTS = [("LLM",8000),("STT",8100),("TTS",8200)]
    tunnels = {}

    def _start_tunnel(name, port):
        logp = f"/kaggle/working/cf_{name.lower()}.log"
        # truncate log
        open(logp,"w").close()
        proc = subprocess.Popen(
            [cf,"tunnel","--no-autoupdate","--url",f"http://localhost:{port}"],
            stdout=open(logp,"a"), stderr=subprocess.STDOUT,
        )
        url = None
        deadline = time.monotonic() + 90
        while time.monotonic() < deadline:
            time.sleep(2)
            try: data = open(logp).read()
            except Exception: data = ""
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", data)
            if m:
                url = m.group(0)
                break
            if proc.poll() is not None:
                break
        return proc, url

    print("\nLaunching cloudflared tunnels...")
    for name, port in PORTS:
        proc, url = _start_tunnel(name, port)
        tunnels[name] = (proc, port, url)
        print(f"  {name} :{port} → {url}")

    # ── 6. Print URL block for CPU-side to grep ──────────────────────
    print("\n---URLS---")
    for name, (_, _, url) in tunnels.items():
        print(f"{name}_BASE_URL={url}")
    print("---END---", flush=True)

    # ── 7. Keep-alive: restart any dead tunnel ───────────────────────
    print("\nKeep-alive loop started (60s heartbeat)")
    heartbeat = 0
    try:
        while True:
            time.sleep(60)
            heartbeat += 1
            # Check services
            svc_dead = []
            for n, p in [("LLM",llm_proc),("STT",stt_proc),("TTS",tts_proc)]:
                if p is not None and p.poll() is not None:
                    svc_dead.append(n)
            # Check tunnels
            tun_dead = []
            for name, (proc, port, url) in list(tunnels.items()):
                if proc.poll() is not None:
                    tun_dead.append(name)
                    new_proc, new_url = _start_tunnel(name, port)
                    tunnels[name] = (new_proc, port, new_url)
                    print(f"  [restart] {name} tunnel → {new_url}", flush=True)
            if heartbeat % 5 == 0 or svc_dead or tun_dead:
                print(f"[hb {heartbeat*60}s] svc_dead={svc_dead} tun_dead={tun_dead}", flush=True)
                if svc_dead:
                    print("*** critical: service died, aborting serve ***", flush=True)
                    break
    except KeyboardInterrupt:
        print("interrupted")

print("SERVE cell exiting")
